# 💰 Notebook 12: Trading Strategy Backtesting

---

**Author:** Tuhin Bhattacharya  
**Date:** February 2025  
**Series:** Tata Motors Deep Dive (12 of 13)

## Objective

A model that predicts direction is useless if it can't make money. We:
1. Build a **walk-forward backtest** using ML predictions
2. Compare against **Buy & Hold** benchmark
3. Calculate **risk-adjusted metrics** (Sharpe, Sortino, Max Drawdown)
4. Analyze performance across **market regimes**

### The Core Question:
> **Can our ML model beat simply buying Tata Motors and holding?**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
import os, warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
PROCESSED_DIR = '../data/processed'
print('✅ Environment ready')

In [ ]:
# Load data
df = pd.read_csv(os.path.join(PROCESSED_DIR, 'tata_motors_all_features.csv'), index_col=0, parse_dates=True) if os.path.exists(os.path.join(PROCESSED_DIR, 'tata_motors_all_features.csv')) else pd.read_csv(os.path.join(PROCESSED_DIR, 'tata_motors_clean.csv'), index_col=0, parse_dates=True)
if 'Log_Return' not in df.columns:
    df['Log_Return'] = np.log(df['Close'] / df['Close'].shift(1))
df['Target'] = (df['Log_Return'].shift(-1) > 0).astype(int)

sel_file = os.path.join(PROCESSED_DIR, 'selected_features.csv')
if os.path.exists(sel_file):
    selected = [f for f in pd.read_csv(sel_file).iloc[:, 0].tolist() if f in df.columns]
else:
    exclude = ['Target', 'Regime', 'Cluster', 'Log_Return', 'Simple_Return', 'Price_Change']
    selected = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude][:15]

model_df = df[selected + ['Target', 'Close', 'Log_Return']].dropna()
print(f'Data: {model_df.shape}, Features: {len(selected)}')

---

## Part 1: Strategy Design

### Signal Rules:
- **BUY** (position = 1): Model predicts UP with probability > 0.5
- **SELL** (position = 0): Model predicts DOWN
- **Long-only**: No short selling

### Realistic Assumptions:
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Transaction cost | 0.1% | Brokerage + STT + GST |
| Starting capital | ₹100,000 | Standard amount |
| Position sizing | 100% | Fully invested or fully cash |

In [ ]:
TRANSACTION_COST = 0.001
STARTING_CAPITAL = 100000
RETRAIN_PERIOD = 63
MIN_TRAIN = 252

print(f'Transaction cost: {TRANSACTION_COST*100}%')
print(f'Starting capital: ₹{STARTING_CAPITAL:,.0f}')
print(f'Retrain period: {RETRAIN_PERIOD} days')

### Walk-Forward Backtesting:
```
Period 1: [===Train===][Trade Q1]
Period 2: [=====Train=====][Trade Q2]
Period 3: [=======Train=======][Trade Q3]
```
This simulates real-world conditions where you periodically refit your model.

In [ ]:
scaler = StandardScaler()
results = []
current_position = 0

# Auto-adjust MIN_TRAIN for shorter datasets
actual_min_train = min(MIN_TRAIN, max(int(len(model_df) * 0.6), 30))
if len(model_df) < 50:
    print(f'⚠️  Very short dataset ({len(model_df)} rows). Backtest will be limited.')

for i in range(actual_min_train, len(model_df), RETRAIN_PERIOD):
    train_end = i
    test_end = min(i + RETRAIN_PERIOD, len(model_df))
    train_data = model_df.iloc[:train_end]
    test_data = model_df.iloc[train_end:test_end]
    if len(test_data) == 0: break
    
    X_train_scaled = scaler.fit_transform(train_data[selected])
    X_test_scaled = scaler.transform(test_data[selected])
    
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
    model.fit(X_train_scaled, train_data['Target'])
    probs = model.predict_proba(X_test_scaled)[:, 1]
    signals = (probs > 0.5).astype(int)
    
    for j, (idx, row) in enumerate(test_data.iterrows()):
        signal = signals[j]
        cost = TRANSACTION_COST if signal != current_position else 0
        strategy_return = (row['Log_Return'] - cost) if current_position == 1 else -cost
        results.append({'Date': idx, 'Close': row['Close'], 'Signal': signal,
                       'Position': current_position, 'Market_Return': row['Log_Return'],
                       'Strategy_Return': strategy_return, 'Probability': probs[j]})
        current_position = signal

if len(results) == 0:
    print('⚠️  No backtest results — dataset too small for walk-forward.')
    print('     Creating placeholder with buy-and-hold returns.')
    bt = model_df[['Close', 'Log_Return']].copy()
    bt['Signal'] = 1
    bt['Position'] = 1
    bt['Market_Return'] = bt['Log_Return']
    bt['Strategy_Return'] = bt['Log_Return']
    bt['Probability'] = 0.5
else:
    bt = pd.DataFrame(results).set_index('Date')
print(f'Backtest: {bt.index.min().date()} to {bt.index.max().date()}')
print(f'Days: {len(bt)}, Invested: {(bt["Position"]==1).mean()*100:.1f}%')

---

## Part 2: Performance Analysis

In [ ]:
bt['Cum_Market'] = (1 + bt['Market_Return']).cumprod()
bt['Cum_Strategy'] = (1 + bt['Strategy_Return']).cumprod()
bt['Market_Value'] = bt['Cum_Market'] * STARTING_CAPITAL
bt['Strategy_Value'] = bt['Cum_Strategy'] * STARTING_CAPITAL

print(f'Buy & Hold:  ₹{bt["Market_Value"].iloc[-1]:,.0f}')
print(f'ML Strategy: ₹{bt["Strategy_Value"].iloc[-1]:,.0f}')

In [ ]:
def calc_metrics(returns, name):
    total_ret = (1 + returns).prod() - 1
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else 0
    down = returns[returns < 0]
    sortino = (returns.mean() / down.std() * np.sqrt(252)) if len(down) > 0 and down.std() > 0 else 0
    cum = (1 + returns).cumprod()
    max_dd = ((cum - cum.expanding().max()) / cum.expanding().max()).min()
    win = (returns > 0).mean()
    return {'Strategy': name, 'Total Return': f'{total_ret*100:+.1f}%', 'Ann. Return': f'{ann_ret*100:+.1f}%',
            'Ann. Vol': f'{ann_vol*100:.1f}%', 'Sharpe': f'{sharpe:.2f}', 'Sortino': f'{sortino:.2f}',
            'Max DD': f'{max_dd*100:.1f}%', 'Win Rate': f'{win*100:.1f}%'}

metrics = pd.DataFrame([calc_metrics(bt['Market_Return'], 'Buy & Hold'),
                       calc_metrics(bt['Strategy_Return'], 'ML Strategy')]).set_index('Strategy')
print('\nPERFORMANCE COMPARISON')
print(metrics)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 14), gridspec_kw={'height_ratios': [3, 1, 1]})
ax = axes[0]
ax.plot(bt.index, bt['Market_Value'], color='gray', linewidth=1.5, label='Buy & Hold', alpha=0.7)
ax.plot(bt.index, bt['Strategy_Value'], color='#2ECC71', linewidth=2, label='ML Strategy')
ax.axhline(STARTING_CAPITAL, color='black', linestyle=':', alpha=0.3)
ax.set_title(f'Equity Curve (Start: ₹{STARTING_CAPITAL:,.0f})', fontsize=14, fontweight='bold')
ax.set_ylabel('Portfolio Value (INR)'); ax.legend(fontsize=12); ax.grid(True, alpha=0.3)

ax = axes[1]
for label, col, color in [('Buy & Hold', 'Market_Return', 'gray'), ('Strategy', 'Strategy_Return', '#E74C3C')]:
    cum = (1 + bt[col]).cumprod(); peak = cum.expanding().max()
    dd = (cum - peak) / peak * 100
    ax.fill_between(bt.index, dd, 0, color=color, alpha=0.3, label=label)
ax.set_title('Drawdown', fontweight='bold'); ax.set_ylabel('DD (%)'); ax.legend(fontsize=10)

ax = axes[2]
ax.fill_between(bt.index, bt['Position'], 0, step='pre', alpha=0.3, color='#3498DB')
ax.set_title('Position (1=Invested, 0=Cash)', fontweight='bold'); ax.set_ylim(-0.1, 1.1)
plt.tight_layout(); plt.show()

**Equity Curve Interpretation:**
- The ML strategy should have lower drawdowns (goes to cash during predicted downturns)
- Buy & Hold captures all upside AND downside
- The position chart shows how often the model is invested

---

## Part 3: Trade Analysis

In [ ]:
trades = bt['Signal'].diff().ne(0).sum()
print('TRADE STATISTICS')
print(f'Signal changes: {trades}')
print(f'Avg trades/month: {trades/(len(bt)/21):.1f}')

In [ ]:
invested = bt[bt['Position'] == 1]
cash = bt[bt['Position'] == 0]
print('\nWhen INVESTED:')
if len(invested) > 0:
    print(f'  Avg return: {invested["Market_Return"].mean()*100:+.4f}%/day')
    print(f'  Win rate: {(invested["Market_Return"] > 0).mean()*100:.1f}%')
print('\nWhen in CASH:')
if len(cash) > 0:
    print(f'  Market avg: {cash["Market_Return"].mean()*100:+.4f}%/day')
    avoided = cash[cash['Market_Return'] < 0]['Market_Return'].sum()
    missed = cash[cash['Market_Return'] > 0]['Market_Return'].sum()
    print(f'  Loss avoided: {avoided*100:+.2f}%')
    print(f'  Gain missed: {missed*100:+.2f}%')

In [ ]:
# Monthly returns heatmap
bt['Year'] = bt.index.year; bt['Month'] = bt.index.month
monthly = bt.groupby(['Year', 'Month'])['Strategy_Return'].apply(lambda x: (1+x).prod()-1).unstack()
fig, ax = plt.subplots(figsize=(14, max(4, len(monthly)*0.6)))
sns.heatmap(monthly*100, annot=True, fmt='.1f', cmap='RdYlGn', center=0, ax=ax, linewidths=0.5,
           xticklabels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'])
ax.set_title('Monthly Strategy Returns (%)', fontweight='bold', fontsize=13)
plt.tight_layout(); plt.show()

---

## Part 4: Regime Performance

In [ ]:
def assign_regime(date):
    if date < pd.Timestamp('2020-03-01'): return 'Pre-COVID'
    elif date < pd.Timestamp('2020-05-01'): return 'COVID Crash'
    elif date < pd.Timestamp('2022-01-01'): return 'Recovery'
    elif date < pd.Timestamp('2024-10-01'): return 'Post-COVID'
    else: return 'Oct 2024'

bt['Regime'] = bt.index.map(assign_regime)
regime_perf = []
for regime in ['Pre-COVID', 'COVID Crash', 'Recovery', 'Post-COVID', 'Oct 2024']:
    mask = bt['Regime'] == regime
    if mask.sum() > 5:
        s = bt.loc[mask, 'Strategy_Return']; m = bt.loc[mask, 'Market_Return']
        regime_perf.append({'Regime': regime, 'Days': mask.sum(),
                          'Market%': f'{m.mean()*252*100:+.1f}', 'Strategy%': f'{s.mean()*252*100:+.1f}',
                          'Invested': f'{bt.loc[mask,"Position"].mean()*100:.0f}%'})
print('REGIME PERFORMANCE')
print(pd.DataFrame(regime_perf).to_string(index=False))

In [ ]:
rp = pd.DataFrame(regime_perf).set_index('Regime')
rp['Market%'] = rp['Market%'].astype(float); rp['Strategy%'] = rp['Strategy%'].astype(float)
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(rp)); width = 0.35
ax.bar(x - width/2, rp['Market%'], width, label='Buy & Hold', color='gray', alpha=0.7)
ax.bar(x + width/2, rp['Strategy%'], width, label='ML Strategy', color='#2ECC71', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(rp.index, rotation=15)
ax.set_ylabel('Ann. Return (%)'); ax.set_title('Strategy vs Buy & Hold by Regime', fontweight='bold', fontsize=14)
ax.legend(fontsize=12); ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout(); plt.show()

**Regime Insights:**
- The strategy adds most value during **crash periods** (going to cash)
- During strong uptrends, Buy & Hold may outperform
- The real value is **risk management**

---

## Part 5: What-If Analysis

In [ ]:
rets = bt['Market_Return'].sort_values()
print('WHAT-IF: AVOIDING THE WORST DAYS')
for n in [5, 10, 20, 50]:
    modified = bt['Market_Return'].copy()
    modified.loc[rets.head(n).index] = 0
    total = (1 + modified).prod()
    bh = (1 + bt['Market_Return']).prod()
    print(f'Avoid worst {n:2d} days: {(total-1)*100:+.1f}% vs B&H {(bh-1)*100:+.1f}%')
print('\n→ Avoiding the worst 10 days dramatically changes returns.')

In [ ]:
bt.to_csv(os.path.join(PROCESSED_DIR, 'backtest_results.csv'))
metrics.to_csv(os.path.join(PROCESSED_DIR, 'strategy_metrics.csv'))
print('✅ Saved: backtest_results.csv, strategy_metrics.csv')

---

## Summary

### 🔑 Key Findings:

1. **Risk reduction > Return enhancement** — ML strategy's primary value is lower drawdowns
2. **Transaction costs matter** — High-frequency trading erodes edge
3. **Regime awareness helps** — Best during crash periods
4. **Worst-days analysis** — Avoiding 10 worst days transforms returns
5. **Buy & Hold is hard to beat** — Most active strategies underperform after costs

### Practical Implications:
- Use ML as a **risk indicator**, not pure trading signal
- Reduce position size when model predicts DOWN
- Combine with sentiment and technical signals
- Rebalance quarterly to reduce transaction costs

---
*Next: Notebook 13 — Final Synthesis*